# AMOS22 → L3 VFA/PMA Tek Tık Pipeline (TS + C2C, 60 Epok)

Bu notebook, Google Colab GPU üzerinde AMOS22 verilerinizle sıfırdan en doğru VFA/PMA hesaplaması ve eğitim akışını adım adım otomatikleştirir.

İş Akışı:
- Ortam kurulumu (GPU, Drive, repo, bağımlılıklar)
- AMOS22 NIfTI → DICOM dönüşümü (plastimatch tercihli)
- Öğretmen çıkarımı: TotalSegmentator (TS) + (varsa) C2C
- L3 seçimi, iç abdominal fasya ve psoas segmentasyonu, VFA/PMA hesapları
- Parametre/preset ayarı (radyolog uyumluluğu için kritik HU ve fasya bantları)
- 60 epok eğitim (GPU), kayıt ve görselleştirme
- Değerlendirme: optimize_params, tek vaka headless eval

Hedefler:
- VFA MAE < 50 mm², PMA MAE < 15 mm²
- Bias |±5%| içinde
- Pearson r > 0.90

In [ ]:
#@title Ortam: GPU, Drive, repo ve bağımlılıklar (Tek Tık)
USE_GPU = True  #@param {type:"boolean"}
REPO_URL = "https://github.com/alpogras23/L3-SO-.git"  #@param {type:"string"}
BRANCH   = "psoas-improvement"  #@param {type:"string"}

from google.colab import drive
drive.mount('/content/drive')

# GPU kontrol
!nvidia-smi || echo 'GPU görünmüyor (CPU ile devam edilebilir).'

# Temel paketler
!pip -q install -U pip setuptools wheel
# Torch + ekosistem (Colab CUDA sürümüne uyacak şekilde iki denemeli)
!pip -q install 'torch>=2.2' 'torchvision>=0.17' 'torchaudio>=2.2' --index-url https://download.pytorch.org/whl/cu121 || pip -q install 'torch>=2.2' 'torchvision>=0.17' 'torchaudio>=2.2'
# Tıbbi görüntüleme + CV
!pip -q install monai pytorch-lightning==2.4.* SimpleITK nibabel opencv-python-headless pydicom scikit-image tqdm rich seaborn pandas
# TotalSegmentator + nnUNetv2
!pip -q install TotalSegmentator==1.7.4 nnunetv2==2.3.1

%cd /content
!rm -rf L3_SO_ANALYSIS
!git clone {REPO_URL} L3_SO_ANALYSIS
%cd L3_SO_ANALYSIS
!git checkout {BRANCH}
print('Kurulum tamamlandı.')

In [ ]:
#@title Yol Değişkenleri ve Düğmeler
# AMOS22 kök dizininizi ve çıktı klasörlerini Drive üzerinde ayarlayın.
AMOS22_ROOT = "/content/drive/MyDrive/AMOS22"  #@param {type:"string"}
DICOM_OUT   = "/content/drive/MyDrive/AMOS22_DICOM"  #@param {type:"string"}
TEACHERS_OUT= "/content/drive/MyDrive/L3_teachers_out"  #@param {type:"string"}
QA_OUT      = "/content/drive/MyDrive/L3_QA_OUT"  #@param {type:"string"}
CKPT_DIR    = "/content/drive/MyDrive/L3_ckpts"  #@param {type:"string"}
LABELS_CFG  = "/content/L3_SO_ANALYSIS/scripts/labels_config_example.json"  #@param {type:"string"}
SITE_PRESET = "eval"  #@param ["fast", "full", "eval"]
RUN_TS      = True  #@param {type:"boolean"}
RUN_C2C     = False #@param {type:"boolean"}
C2C_WEIGHTS = ""  #@param {type:"string"}

import os
for p in [DICOM_OUT, TEACHERS_OUT, QA_OUT, CKPT_DIR]:
    os.makedirs(p, exist_ok=True)
print('Yollar hazır.')

In [ ]:
#@title AMOS22 Dönüşüm: NIfTI → DICOM (plastimatch tercihli)
USE_PLASTIMATCH = True  #@param {type:"boolean"}

import subprocess
import shutil
%cd /content/L3_SO_ANALYSIS

def ensure_plastimatch():
    try:
        r = subprocess.run(['plastimatch', '--version'], capture_output=True)
        if r.returncode == 0:
            print('Plastimatch bulundu.')
            return True
    except FileNotFoundError:
        pass
    if USE_PLASTIMATCH:
        print('Plastimatch kuruluyor...')
        !sudo apt-get update -qq
        !sudo apt-get install -y plastimatch || echo 'Kurulum başarısız, SimpleITK fallback kullanılacak.'
        try:
            r = subprocess.run(['plastimatch', '--version'], capture_output=True)
            return r.returncode == 0
        except FileNotFoundError:
            return False
    return False

pm_ok = ensure_plastimatch()
prefer = 'true' if (USE_PLASTIMATCH and pm_ok) else 'false'
print('Prefer plastimatch:', prefer)
!python scripts/prepare_amos_dicom.py --amos-root "{AMOS22_ROOT}" --out "{DICOM_OUT}" --prefer-plastimatch {prefer}

In [ ]:
#@title Öğretmen Çıkarımı: TotalSegmentator (TS) + C2C (varsa)
%cd /content/L3_SO_ANALYSIS
extra = ''
if RUN_C2C and C2C_WEIGHTS:
    extra = f' --c2c-weights "{C2C_WEIGHTS}"'
cmd = f'python scripts/run_teachers.py --dicom-root "{DICOM_OUT}" --out "{TEACHERS_OUT}" --labels-cfg "{LABELS_CFG}" --use-ts {str(RUN_TS).lower()} --use-c2c {str(RUN_C2C).lower()}{extra}'
print('Çalışan komut:', cmd)
!bash -lc 


In [ ]:
#@title Preset seçimi ve settings.json acil doğruluk override
%cd /content/L3_SO_ANALYSIS
!python scripts/select_preset.py {SITE_PRESET}

from pathlib import Path
import json
cfg_path = Path('settings.json')
cfg = {}
if cfg_path.exists():
    try:
        cfg = json.loads(cfg_path.read_text())
    except Exception as e:
        print('Mevcut settings.json okunamadı, baştan yazılacak:', e)
cfg.update({
  'VB_HU_MIN': 150,
  'VB_HU_MAX': 4000,
  'FASCIA_RING_BAND_MM': 20,
  'GUARD_BAND_MM': 30,
  'VFA_HU_LOW': -180,
  'VFA_HU_HIGH': -20,
  'PSOAS_HU_MIN': -10,
  'PSOAS_HU_MAX': 100,
  'OVERLAY_ALPHA': 0.75,
  'OVERLAY_NORMALIZE_HU': True,
  'OVERLAY_SHOW_CONFIDENCE': True,
  'OVERLAY_SHOW_MEASUREMENTS': True,
  'OVERLAY_VERTEBRA_OUTLINE': True
})
cfg_path.write_text(json.dumps(cfg, indent=2))
print('settings.json güncellendi.')

In [ ]:
#@title L3 Seçimi, Fasya ve Psoas; VFA/PMA Hesapları + QA Overlays
%cd /content/L3_SO_ANALYSIS
!python scripts/run_amos_multiteacher_pipeline.py --dicom-root "{DICOM_OUT}" --teachers-root "{TEACHERS_OUT}" --qa-out "{QA_OUT}" --labels-cfg "{LABELS_CFG}" --save-overlays --overlay-high-quality

# Olası manifest konumlarını dene
import glob, os
candidates = [
    os.path.join(TEACHERS_OUT, 'manifest.csv'),
    os.path.join(TEACHERS_OUT, 'train_manifest.csv'),
    os.path.join('/content/drive/MyDrive', 'L3_training_manifest.csv')
]
TRAIN_CSV = None
for c in candidates:
    if os.path.exists(c):
        TRAIN_CSV = c
        break
print('Seçilen TRAIN_CSV:', TRAIN_CSV)

In [ ]:
#@title 60 Epok Eğitim (GPU)
# Eğer TRAIN_CSV None ise, TEACHERS_OUT içinden uygun bir manifest üretmeniz gerekebilir.
if 'TRAIN_CSV' not in globals() or TRAIN_CSV is None:
    raise SystemExit('TRAIN_CSV bulunamadı. Manifest oluşturma adımını kontrol edin.')

%cd /content/L3_SO_ANALYSIS
IMG_ROOT  = DICOM_OUT
MASK_ROOT = TEACHERS_OUT
EPOCHS    = 60
!python -m psoas_ml.multiteacher_train --csv "{TRAIN_CSV}" --img_root "{IMG_ROOT}" --mask_root "{MASK_ROOT}" --epochs {EPOCHS} --ckpt_dir "{CKPT_DIR}"
print('Eğitim tamamlandı (60 epok).')

In [ ]:
#@title Eğitim Görselleştirme ve Özet
%cd /content/L3_SO_ANALYSIS
!python scripts/visualize_training.py --log_dir "{CKPT_DIR}" --out "{CKPT_DIR}/training_summary.json"
print('Görselleştirme ve özet tamamlandı.')

In [ ]:
#@title Parametre Optimizasyonu (Ground Truth ile)
GT_CSV = "/content/drive/MyDrive/ground_truth.csv"  #@param {type:"string"}
CASES_DIR = DICOM_OUT
%cd /content/L3_SO_ANALYSIS
!python desktop_project/optimize_params.py --ref "{GT_CSV}" --cases-dir "{CASES_DIR}" --random 50 --bias-penalty 0.2 --pearson-weight 0.1
print('Parametre taraması tamamlandı.')

In [ ]:
#@title Tek Vaka Headless Eval (opsiyonel)
SAMPLE_DCM = "/content/drive/MyDrive/sample.dcm"  #@param {type:"string"}
OUT_PNG    = "/content/drive/MyDrive/sample_overlay.png"  #@param {type:"string"}
HEIGHT_M   = 1.75  #@param {type:"number"}
SEX        = 'M'   #@param ["M", "F"]
%cd /content/L3_SO_ANALYSIS
!python tools/eval_one.py "{SAMPLE_DCM}" "{OUT_PNG}" {HEIGHT_M} {SEX}
print('Tek vaka eval tamamlandı.')

## Sorun Giderme İpuçları
- Plastimatch kurulumunda izin/depoloji sorunu: USE_PLASTIMATCH=False ile SimpleITK fallback kullanın.
- TS indirme/çalışma hatası: Runtime'ı yeniden başlatın ve paket sürümlerini kilitleyin (TotalSegmentator==1.7.4, nnunetv2==2.3.1).
- VFA/PMA aşırı düşük/yüksek: settings.json'daki VFA_HU_LOW/HIGH ve PSOAS_HU_MIN/MAX ile FASCIA_* parametrelerini genişletin.
- Vertebra merkez ofsetleri: VB_HU_MIN/MAX ve morfoloji alan eşiklerini optimize edin; QA JSON'da VB_CENTER_OFFSET_RATIO değerini kontrol edin.
- QA görselleri ve JSON'ları mutlaka inceleyin; CONFIDENCE_SCORE < 0.75 ve LEAK_FLAG=1 olanları manuel gözden geçirin.